# Lecture 3: Reading and Writing Structures

## Overview
**Questions**
- How do I read and write structure files in different formats?
- Which file formats does ASE support?
- How do I visualise a sequence of structures?

**Objectives**
- Read a CIF file and inspect the resulting `Atoms` object
- Write structures to different formats (XYZ, POSCAR, CIF)
- Work with trajectories


## Reading structures with `ase.io.read`

ASE can read and write over 50 file formats. The primary function is `ase.io.read`, which auto-detects the format from the file extension.

Common formats you will encounter:

| Format | Extension | Used by |
|--------|-----------|---------|
| CIF | `.cif` | Crystallographic databases (ICSD, COD, MP) |
| POSCAR/CONTCAR | (no extension) | VASP |
| XYZ | `.xyz` | General; Ovito |
| extXYZ | `.xyz` | CASTEP, QUIP, many ML codes |
| Quantum ESPRESSO | `.pwi` / `.pwo` | QE input/output |
| FHI-aims | `geometry.in` | FHI-aims |
| Trajectory | `.traj` | ASE native format |


In [ ]:
import ase.io
from ase.io import read, write
from ase import Atoms
import numpy as np

# We'll build a structure and save it rather than reading from disk,
# so this notebook runs without external files.

# Build wurtzite GaN using ASE's build module
from ase.build import bulk

gan_wz = bulk('GaN', crystalstructure='wurtzite', a=3.19, c=5.19)
print(f"Wurtzite GaN: {len(gan_wz)} atoms, formula {gan_wz.get_chemical_formula()}")
print(f"Cell (Å):\n{gan_wz.cell}\n")

# Build diamond structure
diamond = bulk('C', 'diamond', a=3.57)
print(f"Diamond: {len(diamond)} atoms")


### Writing to different formats


In [ ]:
import os

# Write to XYZ (simple, universal)
write('/tmp/GaN_wurtzite.xyz', gan_wz)

# Write to VASP POSCAR format
write('/tmp/POSCAR_GaN', gan_wz, format='vasp')

# Write to CIF
write('/tmp/GaN_wurtzite.cif', gan_wz, format='cif')

print("Written files:")
for f in ['GaN_wurtzite.xyz', 'POSCAR_GaN', 'GaN_wurtzite.cif']:
    path = f'/tmp/{f}'
    if os.path.exists(path):
        print(f"  {f}: {os.path.getsize(path)} bytes")


In [ ]:
# Read back the XYZ file
gan_readback = read('/tmp/GaN_wurtzite.xyz')
print("Read back from XYZ:")
print(f"  Formula: {gan_readback.get_chemical_formula()}")
print(f"  Positions match: {np.allclose(gan_wz.positions, gan_readback.positions)}")


### Inspecting XYZ file content

The extended XYZ (extXYZ) format is particularly useful because it stores cell and PBC information in the header line:


In [ ]:
with open('/tmp/GaN_wurtzite.xyz') as f:
    print(f.read())


## Trajectories

A **trajectory** is a sequence of `Atoms` objects, typically snapshots from a molecular dynamics run or a geometry optimisation. ASE has a native `.traj` format for storing these efficiently.


In [ ]:
from ase.io.trajectory import Trajectory
from ase.build import bulk
from ase.calculators.emt import EMT
from ase.optimize import BFGS

# Quick example: optimise Al and record the trajectory
al = bulk('Al', 'fcc', a=4.10)   # slightly off equilibrium
al.calc = EMT()

traj_path = '/tmp/al_opt.traj'
opt = BFGS(al, trajectory=traj_path, logfile='/tmp/al_opt.log')
opt.run(fmax=0.01)

# Read the trajectory back
from ase.io import read as ase_read
traj = ase_read(traj_path, index=':')  # ':' means all frames
print(f"Optimisation took {len(traj)} steps")
print(f"Initial energy:  {traj[0].get_potential_energy():.4f} eV")
print(f"Final energy:    {traj[-1].get_potential_energy():.4f} eV")


In [ ]:
# Plot the energy convergence during optimisation
import matplotlib.pyplot as plt

energies = [atoms.get_potential_energy() for atoms in traj]
plt.figure(figsize=(7, 4))
plt.plot(energies, 'o-', color='steelblue')
plt.xlabel('Optimisation step')
plt.ylabel('Energy (eV)')
plt.title('Al geometry optimisation convergence')
plt.tight_layout()
plt.show()


## The Materials Project and ASE

The [Materials Project](https://materialsproject.org) provides a large database of DFT-computed properties. You can download CIF files directly from the website, or use the `mp-api` Python client:

```python
# pip install mp-api
from mp_api.client import MPRester

with MPRester("YOUR_API_KEY") as mpr:
    # Fetch the structure for diamond cubic GaN (mp-830)
    structure = mpr.get_structure_by_material_id("mp-830")
    # Convert from pymatgen Structure to ASE Atoms
    from pymatgen.io.ase import AseAtomsAdaptor
    atoms = AseAtomsAdaptor.get_atoms(structure)
```

This is a very convenient way to get starting structures for quantum optics materials.

---

## Key Points

- `ase.io.read` and `ase.io.write` handle 50+ file formats
- Format is usually auto-detected from the file extension
- Trajectories store sequences of `Atoms` objects and can be read with `index=':'`
- The Materials Project is a great source of starting structures

## Exercise 3.1

Download the CIF file for hexagonal boron nitride (hBN) from the [Crystallography Open Database](http://www.crystallography.net/cod/) (search for "boron nitride hexagonal"). Read it with ASE and print the lattice parameters and number of atoms in the unit cell.

## Exercise 3.2

Construct a diamond cubic silicon unit cell (a = 5.43 Å), write it to POSCAR format, open the file in a text editor and identify each section. Then read it back and verify the structure is intact.
